# NBA 王朝球星進攻效率與勝率之相關性研究：跨時代對比分析

## 分析題目與動機
在 NBA 歷史中，90 年代的陣地防守、00 年代的巨星單打，到 10 年代的大三分時代，籃球戰術經歷了巨大的變革。作為資料科學的愛好者，我感興趣的是：「個人進攻效率（Efficiency）轉化為團隊勝利（Winning）的機制，在不同時代是否有顯著差異？」

**研究對象：**
* Michael Jordan (90s)：代表高強度防守時代。
* Kobe Bryant (00s)：代表單打戰術巔峰。
* Stephen Curry (15-18)：代表現代三分投射時代。

我將分析這三位球星在贏球與輸球場次中的真實命中率 (True Shooting Percentage, TS%) 作為核心效率指標，探討其與勝率（Win/Loss）的相關性。

In [3]:
# 裝入分析必要使用的套件
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
# 載入原始數據 
# df_play_by_play : Play-by-Play 每場球每個球員的事件檔案 (13M+ rows)
df_play_by_play = pd.read_csv("play_by_play.csv")

In [6]:
df_play_by_play

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,...,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,29600012,0,12,0,1,14:43 PM,12:00,NaN,Start of 1st Period (14:43 PM EST),NaN,...,NaN,NaN,0.0,0,NaN,NaN,NaN,NaN,NaN,0
1,29600012,2,10,0,1,14:50 PM,12:00,Jump Ball O'Neal vs. Kleine: Tip to Cassell,NaN,NaN,...,Suns,PHX,5.0,208,Sam Cassell,1.610613e+09,Phoenix,Suns,PHX,0
2,29600012,3,2,1,1,14:51 PM,11:45,NaN,NaN,MISS Cassell 15' Jump Shot,...,NaN,NaN,0.0,0,NaN,NaN,NaN,NaN,NaN,0
3,29600012,4,4,0,1,14:51 PM,11:43,O'Neal REBOUND (Off:0 Def:1),NaN,NaN,...,NaN,NaN,0.0,0,NaN,NaN,NaN,NaN,NaN,0
4,29600012,5,2,1,1,14:51 PM,11:29,MISS Ceballos 26' 3PT Jump Shot,NaN,NaN,...,NaN,NaN,0.0,0,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13592894,32200001,638,1,79,4,10:54 PM,0:00,Brown 30' 3PT Pullup Jump Shot (35 PTS),NaN,NaN,...,NaN,NaN,0.0,0,NaN,NaN,NaN,NaN,NaN,1
13592895,32200001,639,2,1,4,10:54 PM,0:00,NaN,NaN,MISS Markkanen 24' 3PT Jump Shot,...,NaN,NaN,0.0,0,NaN,NaN,NaN,NaN,NaN,1
13592896,32200001,640,4,0,4,10:54 PM,0:00,NaN,NaN,Lillard REBOUND (Off:1 Def:2),...,NaN,NaN,0.0,0,NaN,NaN,NaN,NaN,NaN,1
13592897,32200001,641,1,79,4,10:54 PM,0:00,NaN,NaN,Lillard 30' 3PT Pullup Jump Shot (26 PTS),...,NaN,NaN,0.0,0,NaN,NaN,NaN,NaN,NaN,1


In [7]:
df_play_by_play.columns

Index(['game_id', 'eventnum', 'eventmsgtype', 'eventmsgactiontype', 'period',
       'wctimestring', 'pctimestring', 'homedescription', 'neutraldescription',
       'visitordescription', 'score', 'scoremargin', 'person1type',
       'player1_id', 'player1_name', 'player1_team_id', 'player1_team_city',
       'player1_team_nickname', 'player1_team_abbreviation', 'person2type',
       'player2_id', 'player2_name', 'player2_team_id', 'player2_team_city',
       'player2_team_nickname', 'player2_team_abbreviation', 'person3type',
       'player3_id', 'player3_name', 'player3_team_id', 'player3_team_city',
       'player3_team_nickname', 'player3_team_abbreviation',
       'video_available_flag'],
      dtype='object')

In [8]:
# df_game : Game 事件資料 (13M+ rows)
df_game = pd.read_csv("game.csv")

In [9]:
df_game

,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,reb_away,ast_away,stl_away,blk_away,tov_away,pf_away,pts_away,plus_minus_away,video_available_away,season_type
0,21946,1610610035,HUS,Toronto Huskies,24600001,1946-11-01 00:00:00,HUS vs. NYK,L,0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,68.0,2,0,Regular Season
1,21946,1610610034,BOM,St. Louis Bombers,24600003,1946-11-02 00:00:00,BOM vs. PIT,W,0,20.0,...,NaN,NaN,NaN,NaN,NaN,25.0,51.0,-5,0,Regular Season
2,21946,1610610032,PRO,Providence Steamrollers,24600002,1946-11-02 00:00:00,PRO vs. BOS,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,53.0,-6,0,Regular Season
3,21946,1610610025,CHS,Chicago Stags,24600004,1946-11-02 00:00:00,CHS vs. NYK,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,22.0,47.0,-16,0,Regular Season
4,21946,1610610028,DEF,Detroit Falcons,24600005,1946-11-02 00:00:00,DEF vs. WAS,L,0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,50.0,17,0,Regular Season
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65693,42022,1610612748,MIA,Miami Heat,42200403,2023-06-07 00:00:00,MIA vs. DEN,L,240,34.0,...,58.0,28.0,3.0,5.0,14.0,18.0,109.0,15,1,Playoffs
65694,42022,1610612748,MIA,Miami Heat,42200404,2023-06-09 00:00:00,MIA vs. DEN,L,240,35.0,...,34.0,26.0,11.0,7.0,8.0,18.0,108.0,13,1,Playoffs
65695,42022,1610612743,DEN,Denver Nuggets,42200405,2023-06-12 00:00:00,DEN vs. MIA,W,240,38.0,...,44.0,18.0,9.0,7.0,8.0,21.0,89.0,-5,1,Playoffs
65696,32022,1610616834,LBN,Team LeBron,32200001,2023-02-19 00:00:00,LBN vs. GNS,L,221,79.0,...,46.0,43.0,8.0,1.0,12.0,2.0,184.0,9,1,All-Star


In [10]:
# df_player : Player 球員基本資訊 (65k+ rows)
df_player = pd.read_csv("player.csv")

In [11]:
df_player

,id,full_name,first_name,last_name,is_active
0,76001,Alaa Abdelnaby,Alaa,Abdelnaby,0
1,76002,Zaid Abdul-Aziz,Zaid,Abdul-Aziz,0
2,76003,Kareem Abdul-Jabbar,Kareem,Abdul-Jabbar,0
3,51,Mahmoud Abdul-Rauf,Mahmoud,Abdul-Rauf,0
4,1505,Tariq Abdul-Wahad,Tariq,Abdul-Wahad,0
...,...,...,...,...,...
4826,1627790,Ante Zizic,Ante,Zizic,0
4827,78647,Jim Zoet,Jim,Zoet,0
4828,78648,Bill Zopf,Bill,Zopf,0
4829,1627826,Ivica Zubac,Ivica,Zubac,1


In [12]:
# 確認我們所需的資訊是否有在資料時間範圍內
# 經確認最新的比賽資料有到2023年6月12日，我們所需的比賽資訊只需到2018年，有符合我們的需求
df_game['game_date'] = pd.to_datetime(df_game['game_date'])

print(df_game['game_date'].max())

2023-06-12 00:00:00


In [20]:
df_player[df_player['full_name'] == 'Stephen Curry']

,id,full_name,first_name,last_name,is_active
929,201939,Stephen Curry,Stephen,Curry,1


In [21]:
# 檢視Game資訊的欄位
df_game

,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,reb_away,ast_away,stl_away,blk_away,tov_away,pf_away,pts_away,plus_minus_away,video_available_away,season_type
0,21946,1610610035,HUS,Toronto Huskies,24600001,1946-11-01,HUS vs. NYK,L,0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,68.0,2,0,Regular Season
1,21946,1610610034,BOM,St. Louis Bombers,24600003,1946-11-02,BOM vs. PIT,W,0,20.0,...,NaN,NaN,NaN,NaN,NaN,25.0,51.0,-5,0,Regular Season
2,21946,1610610032,PRO,Providence Steamrollers,24600002,1946-11-02,PRO vs. BOS,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,53.0,-6,0,Regular Season
3,21946,1610610025,CHS,Chicago Stags,24600004,1946-11-02,CHS vs. NYK,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,22.0,47.0,-16,0,Regular Season
4,21946,1610610028,DEF,Detroit Falcons,24600005,1946-11-02,DEF vs. WAS,L,0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,50.0,17,0,Regular Season
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65693,42022,1610612748,MIA,Miami Heat,42200403,2023-06-07,MIA vs. DEN,L,240,34.0,...,58.0,28.0,3.0,5.0,14.0,18.0,109.0,15,1,Playoffs
65694,42022,1610612748,MIA,Miami Heat,42200404,2023-06-09,MIA vs. DEN,L,240,35.0,...,34.0,26.0,11.0,7.0,8.0,18.0,108.0,13,1,Playoffs
65695,42022,1610612743,DEN,Denver Nuggets,42200405,2023-06-12,DEN vs. MIA,W,240,38.0,...,44.0,18.0,9.0,7.0,8.0,21.0,89.0,-5,1,Playoffs
65696,32022,1610616834,LBN,Team LeBron,32200001,2023-02-19,LBN vs. GNS,L,221,79.0,...,46.0,43.0,8.0,1.0,12.0,2.0,184.0,9,1,All-Star


In [22]:
df_game.columns

Index(['season_id', 'team_id_home', 'team_abbreviation_home', 'team_name_home',
       'game_id', 'game_date', 'matchup_home', 'wl_home', 'min', 'fgm_home',
       'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home', 'fg3_pct_home',
       'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home', 'dreb_home',
       'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home', 'pf_home',
       'pts_home', 'plus_minus_home', 'video_available_home', 'team_id_away',
       'team_abbreviation_away', 'team_name_away', 'matchup_away', 'wl_away',
       'fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away',
       'fg3_pct_away', 'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away',
       'dreb_away', 'reb_away', 'ast_away', 'stl_away', 'blk_away', 'tov_away',
       'pf_away', 'pts_away', 'plus_minus_away', 'video_available_away',
       'season_type'],
      dtype='object')

In [23]:
# 篩選出Game裡我們會需要的欄位
df_game_lite = df_game[['game_id','game_date','team_name_home', 'team_name_away', 'matchup_home', 'pts_home', 'pts_away', 'wl_home']]

In [25]:
# 合併df_player和df_play_by_play去篩選出我們想要球員的資訊
merged_df = pd.merge(df_player, df_play_by_play, left_on='id', right_on='player1_id', how='right')

In [27]:
# 檢視合併是否有成功
# 經檢視能把play_by_play裡每個動作連結到球員名字
merged_df

,id,full_name,first_name,last_name,is_active,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,...,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag,points_earned,is_fga,is_fta
0,NaN,NaN,NaN,NaN,NaN,29600012,0,12,0,1,...,0,NaN,NaN,NaN,NaN,NaN,0,0,False,False
1,406.0,Shaquille O'Neal,Shaquille,O'Neal,0.0,29600012,2,10,0,1,...,208,Sam Cassell,1.610613e+09,Phoenix,Suns,PHX,0,0,False,False
2,208.0,Sam Cassell,Sam,Cassell,0.0,29600012,3,2,1,1,...,0,NaN,NaN,NaN,NaN,NaN,0,0,True,False
3,406.0,Shaquille O'Neal,Shaquille,O'Neal,0.0,29600012,4,4,0,1,...,0,NaN,NaN,NaN,NaN,NaN,0,0,False,False
4,76.0,Cedric Ceballos,Cedric,Ceballos,0.0,29600012,5,2,1,1,...,0,NaN,NaN,NaN,NaN,NaN,0,0,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13592894,1627759.0,Jaylen Brown,Jaylen,Brown,1.0,32200001,638,1,79,4,...,0,NaN,NaN,NaN,NaN,NaN,1,3,True,False
13592895,1628374.0,Lauri Markkanen,Lauri,Markkanen,1.0,32200001,639,2,1,4,...,0,NaN,NaN,NaN,NaN,NaN,1,0,True,False
13592896,203081.0,Damian Lillard,Damian,Lillard,1.0,32200001,640,4,0,4,...,0,NaN,NaN,NaN,NaN,NaN,1,0,False,False
13592897,203081.0,Damian Lillard,Damian,Lillard,1.0,32200001,641,1,79,4,...,0,NaN,NaN,NaN,NaN,NaN,1,3,True,False


In [28]:
# 合併merged_df和df_game去篩選出我們想要的比賽日期區間
merged_df_game = pd.merge(merged_df, df_game_lite, left_on='game_id', right_on='game_id', how='left')

In [29]:
# 先從我最愛的咖哩大神開始篩選起，篩選出咖哩大神在2015年到2018年所有打得球賽資訊
merged_df_game_curry_2015_2018 = merged_df_game[(merged_df_game['full_name'] == 'Stephen Curry') & 
                                     (merged_df_game['game_date'] >= '2015/01/01') & 
                                    (merged_df_game['game_date'] <= '2018/12/31')]

In [30]:
# 咖哩大神在這四年總共打了342場球賽
merged_df_game_curry_2015_2018['game_id'].nunique()

342

In [31]:
# 為了要計算勝利時和輸球時的平均得分效率，首先必須把這342場球分成勝利和失敗，接下來再去計算每一場Curry的得分效率，加總後取平均
# 為了要篩選出完整的勝場資訊，必須把主場和客場的勝利資訊都篩出來才能完成
win_curry_game = merged_df_game_curry_2015_2018[
    ((merged_df_game_curry_2015_2018['team_name_home'] == 'Golden State Warriors') & 
     (merged_df_game_curry_2015_2018['wl_home'] == 'W')) 
    |  
    ((merged_df_game_curry_2015_2018['team_name_away'] == 'Golden State Warriors') & 
     (merged_df_game_curry_2015_2018['wl_home'] == 'L'))]

In [32]:
# 檢視是否都是咖哩大神贏的資訊
# 經檢視，都是贏的球賽資訊
win_curry_game

,id,full_name,first_name,last_name,is_active,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,...,points_earned,is_fga,is_fta,game_date,team_name_home,team_name_away,matchup_home,pts_home,pts_away,wl_home
8683599,201939.0,Stephen Curry,Stephen,Curry,1.0,21400493,4,1,79,1,...,3,True,False,2015-01-02,Golden State Warriors,Toronto Raptors,GSW vs. TOR,126.0,105.0,W
8683611,201939.0,Stephen Curry,Stephen,Curry,1.0,21400493,16,3,11,1,...,1,False,True,2015-01-02,Golden State Warriors,Toronto Raptors,GSW vs. TOR,126.0,105.0,W
8683612,201939.0,Stephen Curry,Stephen,Curry,1.0,21400493,18,3,12,1,...,1,False,True,2015-01-02,Golden State Warriors,Toronto Raptors,GSW vs. TOR,126.0,105.0,W
8683619,201939.0,Stephen Curry,Stephen,Curry,1.0,21400493,26,4,0,1,...,0,False,False,2015-01-02,Golden State Warriors,Toronto Raptors,GSW vs. TOR,126.0,105.0,W
8683620,201939.0,Stephen Curry,Stephen,Curry,1.0,21400493,27,2,42,1,...,0,True,False,2015-01-02,Golden State Warriors,Toronto Raptors,GSW vs. TOR,126.0,105.0,W
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11096580,201939.0,Stephen Curry,Stephen,Curry,1.0,21800548,575,4,0,4,...,0,False,False,2018-12-31,Phoenix Suns,Golden State Warriors,PHX vs. GSW,109.0,132.0,L
11096586,201939.0,Stephen Curry,Stephen,Curry,1.0,21800548,588,5,1,4,...,0,False,False,2018-12-31,Phoenix Suns,Golden State Warriors,PHX vs. GSW,109.0,132.0,L
11096590,201939.0,Stephen Curry,Stephen,Curry,1.0,21800548,595,1,1,4,...,3,True,False,2018-12-31,Phoenix Suns,Golden State Warriors,PHX vs. GSW,109.0,132.0,L
11096600,201939.0,Stephen Curry,Stephen,Curry,1.0,21800548,606,2,1,4,...,0,True,False,2018-12-31,Phoenix Suns,Golden State Warriors,PHX vs. GSW,109.0,132.0,L


In [33]:
# 在342場球賽內，咖哩大神總共贏了276場，勝率高達驚人的81%
win_curry_game['game_id'].nunique()

276

In [34]:
# 再把輸的場次篩選出來
loss_curry_game = merged_df_game_curry_2015_2018[
    ((merged_df_game_curry_2015_2018['team_name_home'] == 'Golden State Warriors') & 
     (merged_df_game_curry_2015_2018['wl_home'] == 'L')) 
    |  
    ((merged_df_game_curry_2015_2018['team_name_away'] == 'Golden State Warriors') & 
     (merged_df_game_curry_2015_2018['wl_home'] == 'W'))]

In [35]:
loss_curry_game

,id,full_name,first_name,last_name,is_active,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,...,points_earned,is_fga,is_fta,game_date,team_name_home,team_name_away,matchup_home,pts_home,pts_away,wl_home
8725510,201939.0,Stephen Curry,Stephen,Curry,1.0,21400593,8,2,79,1,...,0,True,False,2015-01-16,Oklahoma City Thunder,Golden State Warriors,OKC vs. GSW,127.0,115.0,W
8725525,201939.0,Stephen Curry,Stephen,Curry,1.0,21400593,25,5,2,1,...,0,False,False,2015-01-16,Oklahoma City Thunder,Golden State Warriors,OKC vs. GSW,127.0,115.0,W
8725527,201939.0,Stephen Curry,Stephen,Curry,1.0,21400593,27,1,78,1,...,2,True,False,2015-01-16,Oklahoma City Thunder,Golden State Warriors,OKC vs. GSW,127.0,115.0,W
8725529,201939.0,Stephen Curry,Stephen,Curry,1.0,21400593,29,4,0,1,...,0,False,False,2015-01-16,Oklahoma City Thunder,Golden State Warriors,OKC vs. GSW,127.0,115.0,W
8725533,201939.0,Stephen Curry,Stephen,Curry,1.0,21400593,33,1,80,1,...,2,True,False,2015-01-16,Oklahoma City Thunder,Golden State Warriors,OKC vs. GSW,127.0,115.0,W
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11081526,201939.0,Stephen Curry,Stephen,Curry,1.0,21800516,701,8,0,4,...,0,False,False,2018-12-27,Golden State Warriors,Portland Trail Blazers,GSW vs. POR,109.0,110.0,L
11081543,201939.0,Stephen Curry,Stephen,Curry,1.0,21800516,734,2,1,5,...,0,True,False,2018-12-27,Golden State Warriors,Portland Trail Blazers,GSW vs. POR,109.0,110.0,L
11081548,201939.0,Stephen Curry,Stephen,Curry,1.0,21800516,739,4,0,5,...,0,False,False,2018-12-27,Golden State Warriors,Portland Trail Blazers,GSW vs. POR,109.0,110.0,L
11081552,201939.0,Stephen Curry,Stephen,Curry,1.0,21800516,747,2,47,5,...,0,True,False,2018-12-27,Golden State Warriors,Portland Trail Blazers,GSW vs. POR,109.0,110.0,L


In [36]:
# 接下來我們要by game_id去計算Curry每場的得分、投籃出手數量、罰球數量
# 投籃效率公式： 得分 / [2 * (投籃出手數 + 0.44 * 罰球出手數)]
# eventmsgtype當中 {FIELD_GOAL_MADE = 1, free_throw_attempt = 3}

In [37]:
# 1. 定義得分計算邏輯 (根據 eventmsgtype 與描述文字)
def calculate_points(row):
    # 投籃得分 (Make)
    if row['eventmsgtype'] == 1:
        desc = str(row['homedescription']) + str(row['visitordescription'])
        return 3 if '3PT' in desc else 2
    # 罰球得分 (Free Throw)
    elif row['eventmsgtype'] == 3:
        desc = str(row['homedescription']) + str(row['visitordescription'])
        return 1 if 'MISS' not in desc.upper() else 0
    return 0

# 2. 預處理：在 DataFrame 中標記得分、FGA 與 FTA 狀態
win_curry_game['points'] = win_curry_game.apply(calculate_points, axis=1)
win_curry_game['is_fga'] = win_curry_game['eventmsgtype'].isin([1, 2])
win_curry_game['is_fta'] = win_curry_game['eventmsgtype'] == 3

# 3. 核心整合：使用一次 groupby 搭配 .agg 算出每場比賽的所有數據
win_curry_game_stats = win_curry_game.groupby('game_id').agg(
    points=('points', 'sum'),
    total_fga=('is_fga', 'sum'),
    total_fta=('is_fta', 'sum')
).reset_index()

# 4. 計算真實命中率 (True Shooting Percentage)
# 公式：PTS / (2 * (FGA + 0.44 * FTA))
win_curry_game_stats['TS%'] = (
    win_curry_game_stats['points'] / 
    (2 * (win_curry_game_stats['total_fga'] + 0.44 * win_curry_game_stats['total_fta']))
)

# 5. 顯示計算結果
print(win_curry_game_stats)

      game_id  points  total_fga  total_fta       TS%
0    11500009      14          7          2  0.888325
1    11500063      19          7          6  0.985477
2    11500099      24         14          2  0.806452
3    11600016      14          8          3  0.751073
4    11600026      13          8          5  0.637255
..        ...     ...        ...        ...       ...
271  41700313      35         23          4  0.706785
272  41700316      29         23          0  0.630435
273  41700317      27         22          0  0.613636
274  41700401      29         23          2  0.607203
275  41700404      37         27          6  0.624157

[276 rows x 5 columns]


/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_67286/2420110336.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  win_curry_game['points'] = win_curry_game.apply(calculate_points, axis=1)
/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_67286/2420110336.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  win_curry_game['is_fga'] = win_curry_game['eventmsgtype'].isin([1, 2])
/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_67286/2420110336.py:16: SettingWithCopyWarning: 
A

In [38]:
# 相同的方法再把輸球的場次計算出來
# 1. 預處理：在 DataFrame 中標記得分、FGA 與 FTA 狀態
loss_curry_game['points'] = loss_curry_game.apply(calculate_points, axis=1)
loss_curry_game['is_fga'] = loss_curry_game['eventmsgtype'].isin([1, 2])
loss_curry_game['is_fta'] = loss_curry_game['eventmsgtype'] == 3

# 2. 核心整合：使用一次 groupby 搭配 .agg 算出每場比賽的所有數據

loss_curry_game_stats = loss_curry_game.groupby('game_id').agg(
    points=('points', 'sum'),
    total_fga=('is_fga', 'sum'),
    total_fta=('is_fta', 'sum')
).reset_index()

# 3. 計算真實命中率 (True Shooting Percentage)
# 公式：PTS / (2 * (FGA + 0.44 * FTA))
loss_curry_game_stats['TS%'] = (
    loss_curry_game_stats['points'] / 
    (2 * (loss_curry_game_stats['total_fga'] + 0.44 * loss_curry_game_stats['total_fta']))
)

# 4. 顯示計算結果
print(loss_curry_game_stats)

     game_id  points  total_fga  total_fta       TS%
0   11500026      30         14         15  0.728155
1   11500073      19         13          0  0.730769
2   11600001       8          6          5  0.487805
3   11700001      11         11          4  0.431034
4   21400593      19         13          5  0.625000
..       ...     ...        ...        ...       ...
57  41500407      17         19          1  0.437243
58  41600404      14         13          5  0.460526
59  41700233      19         19          4  0.457611
60  41700312      16         19          1  0.411523
61  41700314      28         26          2  0.520833

[62 rows x 5 columns]


/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_67286/1957803672.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loss_curry_game['points'] = loss_curry_game.apply(calculate_points, axis=1)
/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_67286/1957803672.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loss_curry_game['is_fga'] = loss_curry_game['eventmsgtype'].isin([1, 2])
/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_67286/1957803672.py:6: SettingWithCopyWarning: 


In [47]:
# 先將所有數據加總
total_pts_win = win_curry_game_stats['points'].sum()
total_fga_win = win_curry_game_stats['total_fga'].sum()
total_fta_win = win_curry_game_stats['total_fta'].sum()

# 代入公式計算總體平均
overall_ts_pct_win = total_pts_win / (2 * (total_fga_win + 0.44 * total_fta_win))

print(f"這段期間勝場的總體平均 TS 為: {overall_ts_pct_win:.4f}")

這段期間勝場的總體平均 TS 為: 0.6695


In [48]:
# 先將所有數據加總
total_pts_loss = loss_curry_game_stats['points'].sum()
total_fga_loss = loss_curry_game_stats['total_fga'].sum()
total_fta_loss = loss_curry_game_stats['total_fta'].sum()

# 代入公式計算總體平均
overall_ts_pct_loss = total_pts_loss / (2 * (total_fga_loss + 0.44 * total_fta_loss))

print(f"這段期間敗場的總體平均 TS 為: {overall_ts_pct_loss:.4f}")

這段期間敗場的總體平均 TS 為: 0.5554


## Curry 王朝的分析發現
* 從Curry角度分析，發現球隊贏球時TS％為0.67，球隊輸球時TS%為0.56，相差了19.6%
* 這也符合一開始的假設，贏場時的TS％高於敗場時的TS%

In [101]:
# 接下來透過相同的步驟把Kobe Bryant的資訊也分析出來
merged_df_game_kobe_2000_2002 = merged_df_game[(merged_df_game['full_name'] == 'Kobe Bryant') & 
                                     (merged_df_game['game_date'] >= '2000/01/01') & 
                                    (merged_df_game['game_date'] <= '2002/12/31')]

In [102]:
# Kobe在這三年總共打了222場球賽
merged_df_game_kobe_2000_2002['game_id'].nunique()

222

In [106]:
# 篩選出Kobe贏的場次
win_kobe_game = merged_df_game_kobe_2000_2002[
    ((merged_df_game_kobe_2000_2002['team_name_home'] == 'Los Angeles Lakers') & 
     (merged_df_game_kobe_2000_2002['wl_home'] == 'W')) 
    |  
    ((merged_df_game_kobe_2000_2002['team_name_away'] == 'Los Angeles Lakers') & 
     (merged_df_game_kobe_2000_2002['wl_home'] == 'L'))]

In [107]:
# 檢驗資料是否皆為贏的場次
# 經過檢視，確實都為贏的場次資訊
win_kobe_game

,id,full_name,first_name,last_name,is_active,game_id,player1_id,player1_name,player2_id,player2_name,...,eventmsgactiontype,homedescription,visitordescription,game_date,team_name_home,team_name_away,matchup_home,pts_home,pts_away,wl_home
1508833,977.0,Kobe Bryant,Kobe,Bryant,0.0,29900440,977,Kobe Bryant,0,NaN,...,1,NaN,MISS Bryant 15' Jump Shot,2000-01-04,Los Angeles Clippers,Los Angeles Lakers,LAC vs. LAL,98.0,122.0,L
1508837,977.0,Kobe Bryant,Kobe,Bryant,0.0,29900440,977,Kobe Bryant,0,NaN,...,6,NaN,Bryant Driving Layup (2 PTS),2000-01-04,Los Angeles Clippers,Los Angeles Lakers,LAC vs. LAL,98.0,122.0,L
1508858,977.0,Kobe Bryant,Kobe,Bryant,0.0,29900440,977,Kobe Bryant,0,NaN,...,0,NaN,Bryant REBOUND (Off:1 Def:0),2000-01-04,Los Angeles Clippers,Los Angeles Lakers,LAC vs. LAL,98.0,122.0,L
1508859,977.0,Kobe Bryant,Kobe,Bryant,0.0,29900440,977,Kobe Bryant,0,NaN,...,1,NaN,MISS Bryant 6' Jump Shot,2000-01-04,Los Angeles Clippers,Los Angeles Lakers,LAC vs. LAL,98.0,122.0,L
1508864,977.0,Kobe Bryant,Kobe,Bryant,0.0,29900440,977,Kobe Bryant,1838,Tyrone Nesby,...,2,Nesby STEAL (1 STL),Bryant Lost Ball Turnover (P1.T1),2000-01-04,Los Angeles Clippers,Los Angeles Lakers,LAC vs. LAL,98.0,122.0,L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2950168,977.0,Kobe Bryant,Kobe,Bryant,0.0,20200431,977,Kobe Bryant,0,NaN,...,5,MISS Bryant Layup,NaN,2002-12-29,Los Angeles Lakers,Toronto Raptors,LAL vs. TOR,104.0,88.0,W
2950172,977.0,Kobe Bryant,Kobe,Bryant,0.0,20200431,977,Kobe Bryant,0,NaN,...,0,Bryant REBOUND (Off:1 Def:8),NaN,2002-12-29,Los Angeles Lakers,Toronto Raptors,LAL vs. TOR,104.0,88.0,W
2950173,977.0,Kobe Bryant,Kobe,Bryant,0.0,20200431,977,Kobe Bryant,283,Lindsey Hunter,...,2,Bryant Lost Ball Turnover (P3.T15),Hunter STEAL (2 STL),2002-12-29,Los Angeles Lakers,Toronto Raptors,LAL vs. TOR,104.0,88.0,W
2950174,977.0,Kobe Bryant,Kobe,Bryant,0.0,20200431,977,Kobe Bryant,0,NaN,...,1,Bryant P.FOUL (P3.T2),NaN,2002-12-29,Los Angeles Lakers,Toronto Raptors,LAL vs. TOR,104.0,88.0,W


In [108]:
# 接著把輸的場次篩選出來
loss_kobe_game = merged_df_game_kobe_2000_2002[
    ((merged_df_game_kobe_2000_2002['team_name_home'] == 'Los Angeles Lakers') & 
     (merged_df_game_kobe_2000_2002['wl_home'] == 'L')) 
    |  
    ((merged_df_game_kobe_2000_2002['team_name_away'] == 'Los Angeles Lakers') & 
     (merged_df_game_kobe_2000_2002['wl_home'] == 'W'))]

In [109]:
# 檢驗資料是否皆為輸的場次
# 經過檢視，確實都為輸的場次資訊
loss_kobe_game

,id,full_name,first_name,last_name,is_active,game_id,player1_id,player1_name,player2_id,player2_name,...,eventmsgactiontype,homedescription,visitordescription,game_date,team_name_home,team_name_away,matchup_home,pts_home,pts_away,wl_home
1538630,977.0,Kobe Bryant,Kobe,Bryant,0.0,29900509,977,Kobe Bryant,0,NaN,...,0,NaN,Bryant REBOUND (Off:0 Def:1),2000-01-14,Indiana Pacers,Los Angeles Lakers,IND vs. LAL,111.0,102.0,W
1538634,977.0,Kobe Bryant,Kobe,Bryant,0.0,29900509,977,Kobe Bryant,0,NaN,...,1,NaN,MISS Bryant 17' Jump Shot,2000-01-14,Indiana Pacers,Los Angeles Lakers,IND vs. LAL,111.0,102.0,W
1538638,977.0,Kobe Bryant,Kobe,Bryant,0.0,29900509,977,Kobe Bryant,0,NaN,...,11,NaN,Bryant Free Throw 1 of 2 (1 PTS),2000-01-14,Indiana Pacers,Los Angeles Lakers,IND vs. LAL,111.0,102.0,W
1538639,977.0,Kobe Bryant,Kobe,Bryant,0.0,29900509,977,Kobe Bryant,0,NaN,...,12,NaN,Bryant Free Throw 2 of 2 (2 PTS),2000-01-14,Indiana Pacers,Los Angeles Lakers,IND vs. LAL,111.0,102.0,W
1538643,977.0,Kobe Bryant,Kobe,Bryant,0.0,29900509,977,Kobe Bryant,406,Shaquille O'Neal,...,8,NaN,Bryant Slam Dunk (4 PTS) (O'Neal 1 AST),2000-01-14,Indiana Pacers,Los Angeles Lakers,IND vs. LAL,111.0,102.0,W
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2940021,977.0,Kobe Bryant,Kobe,Bryant,0.0,20200404,977,Kobe Bryant,0,NaN,...,11,Bryant Free Throw 1 of 2 (26 PTS),NaN,2002-12-25,Los Angeles Lakers,Sacramento Kings,LAL vs. SAC,99.0,105.0,L
2940022,977.0,Kobe Bryant,Kobe,Bryant,0.0,20200404,977,Kobe Bryant,0,NaN,...,12,Bryant Free Throw 2 of 2 (27 PTS),NaN,2002-12-25,Los Angeles Lakers,Sacramento Kings,LAL vs. SAC,99.0,105.0,L
2940028,977.0,Kobe Bryant,Kobe,Bryant,0.0,20200404,977,Kobe Bryant,0,NaN,...,1,MISS Bryant 25' 3PT Jump Shot,NaN,2002-12-25,Los Angeles Lakers,Sacramento Kings,LAL vs. SAC,99.0,105.0,L
2940033,977.0,Kobe Bryant,Kobe,Bryant,0.0,20200404,977,Kobe Bryant,0,NaN,...,1,MISS Bryant 26' 3PT Jump Shot,NaN,2002-12-25,Los Angeles Lakers,Sacramento Kings,LAL vs. SAC,99.0,105.0,L


In [110]:
# 1. 預處理：在 DataFrame 中標記得分、FGA 與 FTA 狀態
win_kobe_game['points'] = win_kobe_game.apply(calculate_points, axis=1)
win_kobe_game['is_fga'] = win_kobe_game['eventmsgtype'].isin([1, 2])
win_kobe_game['is_fta'] = win_kobe_game['eventmsgtype'] == 3

# 2. 核心整合：使用一次 groupby 搭配 .agg 算出每場比賽的所有數據
win_kobe_game_stats = win_kobe_game.groupby('game_id').agg(
    points=('points', 'sum'),
    total_fga=('is_fga', 'sum'),
    total_fta=('is_fta', 'sum')
).reset_index()

# 3. 計算真實命中率 (True Shooting Percentage)
# 公式：PTS / (2 * (FGA + 0.44 * FTA))
win_kobe_game_stats['TS%'] = (
    win_kobe_game_stats['points'] / 
    (2 * (win_kobe_game_stats['total_fga'] + 0.44 * win_kobe_game_stats['total_fta']))
)

# 4. 顯示計算結果
print(win_kobe_game_stats)

      game_id  points  total_fga  total_fta       TS%
0    20000012      14         11          6  0.513196
1    20000049      21         14          7  0.614754
2    20000098      37         29         15  0.519663
3    20000108      31         23          6  0.604524
4    20000124      31         23         12  0.548091
..        ...     ...        ...        ...       ...
152  40000072      24         19          3  0.590551
153  40000084      31         23          8  0.584465
154  40000085      32         30          6  0.490196
155  40000086      19         13         12  0.519694
156  40000087      26         18         11  0.569177

[157 rows x 5 columns]


/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_49772/47279318.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  win_kobe_game['points'] = win_kobe_game.apply(calculate_points, axis=1)
/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_49772/47279318.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  win_kobe_game['is_fga'] = win_kobe_game['eventmsgtype'].isin([1, 2])
/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_49772/47279318.py:4: SettingWithCopyWarning: 
A value is try

In [111]:
# 先將所有數據加總
total_pts_win_kobe = win_kobe_game_stats['points'].sum()
total_fga_win_kobe = win_kobe_game_stats['total_fga'].sum()
total_fta_win_kobe = win_kobe_game_stats['total_fta'].sum()

# 代入公式計算總體平均
overall_ts_pct_win_kobe = total_pts_win_kobe / (2 * (total_fga_win_kobe + 0.44 * total_fta_win_kobe))

print(f"這段期間的總體平均 TS% 為: {overall_ts_pct_win_kobe:.4f}")

這段期間的總體平均 TS% 為: 0.5703


In [112]:
# 1. 預處理：在 DataFrame 中標記得分、FGA 與 FTA 狀態
loss_kobe_game['points'] = loss_kobe_game.apply(calculate_points, axis=1)
loss_kobe_game['is_fga'] = loss_kobe_game['eventmsgtype'].isin([1, 2])
loss_kobe_game['is_fta'] = loss_kobe_game['eventmsgtype'] == 3

# 2. 核心整合：使用一次 groupby 搭配 .agg 算出每場比賽的所有數據
# 這樣就不需要額外執行 pd.merge，程式碼會乾淨很多
loss_kobe_game_stats = loss_kobe_game.groupby('game_id').agg(
    points=('points', 'sum'),
    total_fga=('is_fga', 'sum'),
    total_fta=('is_fta', 'sum')
).reset_index()

# 3. 計算真實命中率 (True Shooting Percentage)
# 公式：PTS / (2 * (FGA + 0.44 * FTA))
loss_kobe_game_stats['TS%'] = (
    loss_kobe_game_stats['points'] / 
    (2 * (loss_kobe_game_stats['total_fga'] + 0.44 * loss_kobe_game_stats['total_fta']))
)

# 4. 顯示計算結果
print(loss_kobe_game_stats)

     game_id  points  total_fga  total_fta       TS%
0   20000019      31         21         13  0.580090
1   20000058      15         16          1  0.456204
2   20000068      32         31          1  0.508906
3   20000140      32         25         10  0.544218
4   20000222      17         16          5  0.467033
..       ...     ...        ...        ...       ...
60  29900923       7         10          4  0.297619
61  29901097      26         30          4  0.409320
62  29901171      16         20          1  0.391389
63  29901185      23         21          4  0.505272
64  40000083      15         22          1  0.334225

[65 rows x 5 columns]


/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_49772/3741405458.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loss_kobe_game['points'] = loss_kobe_game.apply(calculate_points, axis=1)
/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_49772/3741405458.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loss_kobe_game['is_fga'] = loss_kobe_game['eventmsgtype'].isin([1, 2])
/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_49772/3741405458.py:4: SettingWithCopyWarning: 
A va

In [113]:
# 先將所有數據加總
total_pts_loss_kobe = loss_kobe_game_stats['points'].sum()
total_fga_loss_kobe = loss_kobe_game_stats['total_fga'].sum()
total_fta_loss_kobe = loss_kobe_game_stats['total_fta'].sum()

# 代入公式計算總體平均
overall_ts_pct_loss_kobe = total_pts_loss_kobe / (2 * (total_fga_loss_kobe + 0.44 * total_fta_loss_kobe))

print(f"這段期間的總體平均 TS% 為: {overall_ts_pct_loss_kobe:.4f}")

這段期間的總體平均 TS% 為: 0.4907


## Kobe 王朝的分析發現
* 從Kobe角度分析，發現球隊贏球時TS%為0.57，球隊輸球時TS%為0.49，相差了16.3%
* 同樣也符合假設，贏場時的TS％高於敗場時的TS%

In [114]:
# 接下來透過相同的步驟把Michael Jordan的資訊也分析出來
merged_df_game_jordan_1991_1998 = merged_df_game[(merged_df_game['full_name'] == 'Michael Jordan') & (
                                     (merged_df_game['game_date'] >= '1991/01/01') & 
                                    (merged_df_game['game_date'] <= '1993/12/31') | 
                                     (merged_df_game['game_date'] >= '1996/01/01') & 
                                    (merged_df_game['game_date'] <= '1998/12/31') )]

In [126]:
# Michael Jordan這段期間總共打了191場比賽
merged_df_game_jordan_1991_1998['game_id'].nunique()

191

In [127]:
# 篩選出Jordan贏的場次
win_jordan_game = merged_df_game_jordan_1991_1998[
    ((merged_df_game_jordan_1991_1998['team_name_home'] == 'Chicago Bulls') & 
     (merged_df_game_jordan_1991_1998['wl_home'] == 'W')) 
    |  
    ((merged_df_game_jordan_1991_1998['team_name_away'] == 'Chicago Bulls') & 
     (merged_df_game_jordan_1991_1998['wl_home'] == 'L'))]

In [128]:
# 檢驗資料是否皆為贏的場次
# 經過檢視，確實都為贏的場次資訊
win_jordan_game

,id,full_name,first_name,last_name,is_active,game_id,player1_id,player1_name,player2_id,player2_name,...,eventmsgactiontype,homedescription,visitordescription,game_date,team_name_home,team_name_away,matchup_home,pts_home,pts_away,wl_home
2231,893.0,Michael Jordan,Michael,Jordan,0.0,29600001,893,Michael Jordan,0,NaN,...,1,NaN,MISS Jordan 15' Jump Shot,1996-11-01,Boston Celtics,Chicago Bulls,BOS vs. CHI,98.0,107.0,L
2263,893.0,Michael Jordan,Michael,Jordan,0.0,29600001,893,Michael Jordan,0,NaN,...,1,NaN,MISS Jordan 25' 3PT Jump Shot,1996-11-01,Boston Celtics,Chicago Bulls,BOS vs. CHI,98.0,107.0,L
2273,893.0,Michael Jordan,Michael,Jordan,0.0,29600001,893,Michael Jordan,0,NaN,...,0,NaN,Jordan REBOUND (Off:0 Def:1),1996-11-01,Boston Celtics,Chicago Bulls,BOS vs. CHI,98.0,107.0,L
2280,893.0,Michael Jordan,Michael,Jordan,0.0,29600001,893,Michael Jordan,0,NaN,...,1,NaN,Jordan 16' Jump Shot (2 PTS),1996-11-01,Boston Celtics,Chicago Bulls,BOS vs. CHI,98.0,107.0,L
2288,893.0,Michael Jordan,Michael,Jordan,0.0,29600001,893,Michael Jordan,166,Ron Harper,...,1,NaN,Jordan 17' Jump Shot (4 PTS) (Harper 3 AST),1996-11-01,Boston Celtics,Chicago Bulls,BOS vs. CHI,98.0,107.0,L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1019542,893.0,Michael Jordan,Michael,Jordan,0.0,49700088,893,Michael Jordan,0,NaN,...,0,NaN,Jordan REBOUND (Off:0 Def:1),1998-06-14,Utah Jazz,Chicago Bulls,UTA vs. CHI,86.0,87.0,L
1019545,893.0,Michael Jordan,Michael,Jordan,0.0,49700088,893,Michael Jordan,0,NaN,...,11,NaN,Jordan Free Throw 1 of 2 (40 PTS),1998-06-14,Utah Jazz,Chicago Bulls,UTA vs. CHI,86.0,87.0,L
1019546,893.0,Michael Jordan,Michael,Jordan,0.0,49700088,893,Michael Jordan,0,NaN,...,12,NaN,Jordan Free Throw 2 of 2 (41 PTS),1998-06-14,Utah Jazz,Chicago Bulls,UTA vs. CHI,86.0,87.0,L
1019550,893.0,Michael Jordan,Michael,Jordan,0.0,49700088,893,Michael Jordan,0,NaN,...,6,NaN,Jordan Driving Layup (43 PTS),1998-06-14,Utah Jazz,Chicago Bulls,UTA vs. CHI,86.0,87.0,L


In [129]:
# 把Jordan輸的場次篩選出來
loss_jordan_game = merged_df_game_jordan_1991_1998[
    ((merged_df_game_jordan_1991_1998['team_name_home'] == 'Chicago Bulls') & 
     (merged_df_game_jordan_1991_1998['wl_home'] == 'L')) 
    |  
    ((merged_df_game_jordan_1991_1998['team_name_away'] == 'Chicago Bulls') & 
     (merged_df_game_jordan_1991_1998['wl_home'] == 'W'))]

In [130]:
# 檢驗資料是否皆為輸的場次
# 經過檢視，確實都為輸的場次資訊
loss_jordan_game

,id,full_name,first_name,last_name,is_active,game_id,player1_id,player1_name,player2_id,player2_name,...,eventmsgactiontype,homedescription,visitordescription,game_date,team_name_home,team_name_away,matchup_home,pts_home,pts_away,wl_home
107680,893.0,Michael Jordan,Michael,Jordan,0.0,29600261,893,Michael Jordan,0,NaN,...,1,MISS Jordan 18' Jump Shot,NaN,1996-12-07,Chicago Bulls,Miami Heat,CHI vs. MIA,80.0,83.0,L
107689,893.0,Michael Jordan,Michael,Jordan,0.0,29600261,893,Michael Jordan,0,NaN,...,1,MISS Jordan 12' Jump Shot,NaN,1996-12-07,Chicago Bulls,Miami Heat,CHI vs. MIA,80.0,83.0,L
107695,893.0,Michael Jordan,Michael,Jordan,0.0,29600261,893,Michael Jordan,0,NaN,...,1,MISS Jordan 16' Jump Shot,NaN,1996-12-07,Chicago Bulls,Miami Heat,CHI vs. MIA,80.0,83.0,L
107699,893.0,Michael Jordan,Michael,Jordan,0.0,29600261,893,Michael Jordan,0,NaN,...,0,Jordan REBOUND (Off:0 Def:1),NaN,1996-12-07,Chicago Bulls,Miami Heat,CHI vs. MIA,80.0,83.0,L
107711,893.0,Michael Jordan,Michael,Jordan,0.0,29600261,893,Michael Jordan,937,Scottie Pippen,...,1,Jordan 15' Jump Shot (2 PTS) (Pippen 2 AST),NaN,1996-12-07,Chicago Bulls,Miami Heat,CHI vs. MIA,80.0,83.0,L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1019120,893.0,Michael Jordan,Michael,Jordan,0.0,49700087,893,Michael Jordan,0,NaN,...,5,MISS Jordan Layup,NaN,1998-06-12,Chicago Bulls,Utah Jazz,CHI vs. UTA,81.0,83.0,L
1019130,893.0,Michael Jordan,Michael,Jordan,0.0,49700087,893,Michael Jordan,0,NaN,...,11,Jordan Free Throw 1 of 2 (27 PTS),NaN,1998-06-12,Chicago Bulls,Utah Jazz,CHI vs. UTA,81.0,83.0,L
1019131,893.0,Michael Jordan,Michael,Jordan,0.0,49700087,893,Michael Jordan,0,NaN,...,12,Jordan Free Throw 2 of 2 (28 PTS),NaN,1998-06-12,Chicago Bulls,Utah Jazz,CHI vs. UTA,81.0,83.0,L
1019138,893.0,Michael Jordan,Michael,Jordan,0.0,49700087,893,Michael Jordan,0,NaN,...,0,Jordan REBOUND (Off:1 Def:3),NaN,1998-06-12,Chicago Bulls,Utah Jazz,CHI vs. UTA,81.0,83.0,L


In [131]:
# 1. 預處理：在 DataFrame 中標記得分、FGA 與 FTA 狀態
win_jordan_game['points'] = win_jordan_game.apply(calculate_points, axis=1)
win_jordan_game['is_fga'] = win_jordan_game['eventmsgtype'].isin([1, 2])
win_jordan_game['is_fta'] = win_jordan_game['eventmsgtype'] == 3

# 2. 核心整合：使用一次 groupby 搭配 .agg 算出每場比賽的所有數據
win_jordan_game_stats = win_jordan_game.groupby('game_id').agg(
    points=('points', 'sum'),
    total_fga=('is_fga', 'sum'),
    total_fta=('is_fta', 'sum')
).reset_index()

# 3. 計算真實命中率 (True Shooting Percentage)
# 公式：PTS / (2 * (FGA + 0.44 * FTA))
win_jordan_game_stats['TS%'] = (
    win_jordan_game_stats['points'] / 
    (2 * (win_jordan_game_stats['total_fga'] + 0.44 * win_jordan_game_stats['total_fta']))
)

# 4. 顯示計算結果
print(win_jordan_game_stats)

      game_id  points  total_fga  total_fta       TS%
0    29600001      30         22         13  0.541126
1    29600019      27         20          2  0.646552
2    29600035      22         18          1  0.596529
3    29600043      50         33         14  0.638407
4    29600059      15         14          2  0.504032
..        ...     ...        ...        ...       ...
146  49700082      28         25         15  0.443038
147  49700084      37         33         10  0.494652
148  49700085      24         14         11  0.636943
149  49700086      34         27         15  0.505952
150  49700088      45         35         15  0.540865

[151 rows x 5 columns]


/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_49772/1371874353.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  win_jordan_game['points'] = win_jordan_game.apply(calculate_points, axis=1)
/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_49772/1371874353.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  win_jordan_game['is_fga'] = win_jordan_game['eventmsgtype'].isin([1, 2])
/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_49772/1371874353.py:4: SettingWithCopyWarning: 


In [132]:
# 先將所有數據加總
total_pts_win_jordan = win_jordan_game_stats['points'].sum()
total_fga_win_jordan = win_jordan_game_stats['total_fga'].sum()
total_fta_win_jordan = win_jordan_game_stats['total_fta'].sum()

# 代入公式計算總體平均
overall_ts_pct_win_jordan = total_pts_win_jordan / (2 * (total_fga_win_jordan + 0.44 * total_fta_win_jordan))

print(f"這段期間的總體平均 TS% 為: {overall_ts_pct_win_jordan:.4f}")

這段期間的總體平均 TS% 為: 0.5641


In [133]:
# 1. 預處理：在 DataFrame 中標記得分、FGA 與 FTA 狀態
loss_jordan_game['points'] = loss_jordan_game.apply(calculate_points, axis=1)
loss_jordan_game['is_fga'] = loss_jordan_game['eventmsgtype'].isin([1, 2])
loss_jordan_game['is_fta'] = loss_jordan_game['eventmsgtype'] == 3

# 2. 核心整合：使用一次 groupby 搭配 .agg 算出每場比賽的所有數據
loss_jordan_game_stats = loss_jordan_game.groupby('game_id').agg(
    points=('points', 'sum'),
    total_fga=('is_fga', 'sum'),
    total_fta=('is_fta', 'sum')
).reset_index()

# 3. 計算真實命中率 (True Shooting Percentage)
# 公式：PTS / (2 * (FGA + 0.44 * FTA))
loss_jordan_game_stats['TS%'] = (
    loss_jordan_game_stats['points'] / 
    (2 * (loss_jordan_game_stats['total_fga'] + 0.44 * loss_jordan_game_stats['total_fta']))
)

# 4. 顯示計算結果
print(loss_jordan_game_stats)

     game_id  points  total_fga  total_fta       TS%
0   29600261      37         30          6  0.566789
1   29600266      13         17          2  0.363535
2   29600384      34         25         13  0.553385
3   29600548      26         25          9  0.448895
4   29600673      27         24          6  0.506757
5   29600877      36         31          9  0.514874
6   29600912      36         34          5  0.497238
7   29601057      34         30          6  0.520833
8   29601132      18         16          4  0.506757
9   29601155      26         27          4  0.452017
10  29601176      33         22          8  0.646552
11  29700001      30         23         21  0.465261
12  29700057      27         25          7  0.480769
13  29700084      19         17          7  0.473108
14  29700092      28         28         12  0.420673
15  29700151      30         27          6  0.506073
16  29700182      26         25          6  0.470333
17  29700198      26         26          8  0.

/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_49772/2092018521.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loss_jordan_game['points'] = loss_jordan_game.apply(calculate_points, axis=1)
/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_49772/2092018521.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  loss_jordan_game['is_fga'] = loss_jordan_game['eventmsgtype'].isin([1, 2])
/var/folders/v8/0jkf_sjs4qn25v4rrwvlpyb00000gn/T/ipykernel_49772/2092018521.py:4: SettingWithCopyWarnin

In [134]:
# 先將所有數據加總
total_pts_loss_jordan = loss_jordan_game_stats['points'].sum()
total_fga_loss_jordan = loss_jordan_game_stats['total_fga'].sum()
total_fta_loss_jordan = loss_jordan_game_stats['total_fta'].sum()

# 代入公式計算總體平均
overall_ts_pct_loss_jordan = total_pts_loss_jordan / (2 * (total_fga_loss_jordan + 0.44 * total_fta_loss_jordan))

print(f"這段期間的總體平均 TS% 為: {overall_ts_pct_loss_jordan:.4f}")

這段期間的總體平均 TS% 為: 0.4868


## Jordan 王朝的分析發現
* 從Jordan角度分析，發現球隊贏球時TS%為0.56，球隊輸球時TS%為0.49，相差了14.3%
* 最後一個王朝也符合假設，贏場時的TS％高於敗場時的TS%

## 結論與見解
### 透過數據分析，我得出以下幾點觀察：

1. 進攻效率對勝負的「敏感度」逐代增強

* 數據觀察：從 Jordan 王朝 (14.3%)、Kobe 王朝 (16.3%) 到 Curry 王朝 (19.6%)，贏球與輸球之間的效率差距（Efficiency Gap）呈現明顯的上升趨勢。

* 分析意義：這代表隨著時代演進，核心球員的表現對比賽結果的「決定性」越來越高。在 Curry 時代，球員進攻效率的波動與球隊勝負的連結最為緊密，展現了現代籃球「以效率帶動勝利」的特徵。

2. 贏球的「效率門檻」顯著提升 

* 數據觀察：Jordan 王朝與 Kobe 王朝的贏球 TS% 分別為 0.56 與 0.57，兩者差異不大；然而 Curry 王朝的贏球 TS% 躍升至 0.67，整整提升了約 10 個百分點。

* 分析意義：這量化了 NBA 戰術風格的劇烈變遷。在 90 年代與 00 年代，0.56 左右的 TS% 即足以支撐一支冠軍級球隊贏球；但在大三分與空間革命時代，球隊必須達到極致的效率（0.67）才能確保勝利。這反映出當今聯盟「高得分、高期望值」的競爭環境。

3. 王朝贏球邏輯的典範轉移 

* 數據觀察：Jordan 王朝在輸球時仍保有與 Kobe 王朝相同的 0.49 TS%，但其勝負差距 (14.3%) 是三個時期中最小的。

* 分析意義：這暗示了在 Jordan 時代，除了個人得分效率外，可能存在更多元的贏球因子（如高強度的防守、二次進攻或失誤控制）。相對地，Curry 王朝 19.6% 的巨大落差說明了現代籃球的贏球邏輯更趨向「純粹效率化」——當核心球員無法維持超高效率時，球隊更難透過其他手段彌補分差。